# Computer Vision Exam

**Instructions:**
* Run the setup cells first to install/import dependencies.
* Complete the code in the cells marked with `# TODO`.
* You do **not** need to modify the complex boilerplate code (API calls, image loading, etc.).
* Focus on the logic: parsing outputs, coordinate math, and prompting.

In [1]:
# --- SETUP CELL (DO NOT MODIFY) ---
import os
import json
import base64
import requests
import torch
import numpy as np
from PIL import Image, ImageDraw
from dotenv import load_dotenv
from ultralytics import YOLO
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import openai

# Create data folder and download a test image
os.makedirs("exam_data", exist_ok=True)
img_path = "exam_data/test_dog.jpg"
url = "http://images.cocodataset.org/val2017/000000039769.jpg" # Cats and remotes image

if not os.path.exists(img_path):
    raw_img = Image.open(requests.get(url, stream=True).raw)
    raw_img.save(img_path)
    print("Test image downloaded.")
else:
    raw_img = Image.open(img_path)
    print("Test image loaded.")

load_dotenv()
print("Setup complete.")

Test image downloaded.
Setup complete.


### Exercise 1: YOLO Parsing
**Task:** We have run the YOLO model for you. Your job is to extract the **confidence score** and the **bounding box coordinates** from the results.
**Hint:** Remember that tensors need to be converted to scalars using `.item()` to avoid deprecation warnings.

In [ ]:
# 1. Load Model & Run Inference (Provided)
model = YOLO("yolov8n.pt")
results = model(img_path, verbose=False)
result = results[0]

detections = []

# 2. Loop through results
for box in result.boxes:
    # --- TODO: START ---
    # Extract the confidence score as a standard float
    score = # TODO: box.conf... 
    
    # Extract the class ID as a standard int
    class_id = # TODO: box.cls...
    
    # Extract the coordinates [x1, y1, x2, y2] as a list
    coords = # TODO: box.xyxy...
    # --- TODO: END ---
    
    label = model.names[class_id]
    detections.append({"label": label, "score": score, "box": coords})

print(f"First detection: {detections[0]}")

### Exercise 2: Normalization Logic (YOLO Format)
**Task:** To train YOLO, we need coordinates in the format `[x_center, y_center, width, height]` relative to the image size (0.0 to 1.0).
Convert the standard pixel box `[x1, y1, x2, y2]` into this format.

In [ ]:
img_width, img_height = raw_img.size
test_box = [100, 200, 300, 600] # [x1, y1, x2, y2]

def convert_to_yolo_format(box, w, h):
    x1, y1, x2, y2 = box
    
    # --- TODO: START ---
    # Calculate the center X and Y (in pixels)
    center_x = # TODO
    center_y = # TODO
    
    # Calculate box width and height (in pixels)
    box_w = # TODO
    box_h = # TODO
    
    # Normalize everything by dividing by image width/height (result should be 0.0 - 1.0)
    norm_cx = # TODO
    norm_cy = # TODO
    norm_w = # TODO
    norm_h = # TODO
    # --- TODO: END ---
    
    return [norm_cx, norm_cy, norm_w, norm_h]

yolo_box = convert_to_yolo_format(test_box, img_width, img_height)
print(f"YOLO format: {yolo_box}")

### Exercise 3: OWL-ViT Query Setup
**Task:** Prepare the inputs for OWL-ViT. You need to define the text queries correctly (remember the list-of-lists format) and then define the `target_sizes` tensor so the model can scale the boxes back to pixels.

In [ ]:
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

# --- TODO: START ---
# Define a query list containing "cat eyes" and "remote control"
# Remember: processor expects a batch, so it must be a list inside a list: [["a", "b"]]
text_queries = # TODO

# Define target_sizes. It must be a Tensor containing [(height, width)].
# Hint: raw_img.size returns (width, height), so reverse it.
target_sizes = torch.tensor([ # TODO ])
# --- TODO: END ---

inputs = processor(text=text_queries, images=raw_img, return_tensors="pt")

with torch.no_grad():
    outputs = owl_model(**inputs)

results = processor.post_process_object_detection(outputs, threshold=0.1, target_sizes=target_sizes)[0]
print(f"OWL-ViT found {len(results['boxes'])} objects.")

### Exercise 4: VLM Prompt Engineering
**Task:** We want to detect the cat using GPT-4o. Write the **System Prompt** that enforces:
1.  Return format: `[ymin, xmin, ymax, xmax]`
2.  Scaling: `0-1000`
3.  Output type: `JSON`

In [ ]:
# Boilerplate for image encoding (Provided)
def encode_image(image_path):
  with open(image_path, "rb") as f: return base64.b64encode(f.read()).decode('utf-8')
encoded_img = encode_image(img_path)

# --- TODO: START ---
# Write the system prompt string
system_prompt = """
# TODO: Write instructions here.
"""
# --- TODO: END ---

user_prompt = "Detect the cat."

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
try:
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "text", "text": user_prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encoded_img}"}}
            ]}
        ],
        response_format={"type": "json_object"}
    )
    print("Response:", completion.choices[0].message.content)
except Exception as e:
    print("API Call skipped (Mocking result for next exercise). Error:", e)


### Exercise 5: Denormalization (0-1000 -> Pixels)
**Task:** You received a bounding box `[200, 300, 500, 600]` (ymin, xmin, ymax, xmax) in 0-1000 scale.
Convert this back to absolute pixels `[xmin, ymin, xmax, ymax]`.

In [ ]:
vlm_box_norm = [200, 300, 500, 600] # ymin, xmin, ymax, xmax (0-1000 scale)
w, h = raw_img.size

def denormalize_box(box, img_w, img_h):
    ymin, xmin, ymax, xmax = box
    
    # --- TODO: START ---
    # Convert normalized 0-1000 values to absolute pixels
    # Hint: (value / 1000) * total_dimension
    abs_x1 = # TODO
    abs_y1 = # TODO
    abs_x2 = # TODO
    abs_y2 = # TODO
    # --- TODO: END ---
    
    return [abs_x1, abs_y1, abs_x2, abs_y2]

final_box = denormalize_box(vlm_box_norm, w, h)
print(f"Final Pixel Box: {final_box}")

### Exercise 6: Visualization
**Task:** Use `ImageDraw` to draw the `final_box` rectangle on the image.

In [ ]:
vis_img = raw_img.copy()

# --- TODO: START ---
# Initialize the drawing context
draw = # TODO

# Draw the rectangle using final_box
# Use outline="red" and width=3
# TODO: draw.rectangle( ... )
# --- TODO: END ---

display(vis_img.resize((300, 200)))